# 03 — Model Evaluation

Evaluate the trained ResNet classifier: confusion matrix, per-class
metrics, confidence distributions, and failure case analysis.

In [ ]:
import sys
sys.path.append('..')

import numpy as np
import matplotlib.pyplot as plt
import torch
from torch.utils.data import DataLoader, Subset
from pathlib import Path
from collections import defaultdict

from config import (
    NORMALIZED_PATCHES_DIR, DEVICE, CLASS_NAMES, NUM_CLASSES, TRAINING
)
from models import load_classifier
from training.dataset import PatchDataset
from training.augmentations import get_eval_transform

%matplotlib inline
plt.rcParams['figure.figsize'] = (10, 8)
plt.rcParams['figure.dpi'] = 100

## Load Model and Recreate Test Split

Use the same seed and split ratios as training to get the exact same
test set — no data leakage.

In [ ]:
# Load trained model
model, class_names, meta = load_classifier(device=DEVICE)
print(f"Model from epoch {meta['epoch']}")
print(f"Reported test accuracy: {meta.get('test_accuracy', 'N/A')}%")

# Recreate the same test split
import random
random.seed(TRAINING["seed"])

dataset = PatchDataset(NORMALIZED_PATCHES_DIR, transform=get_eval_transform(), class_names=CLASS_NAMES)
n = len(dataset)
indices = list(range(n))
random.shuffle(indices)

train_end = int(TRAINING["train_split"] * n)
val_end = int((TRAINING["train_split"] + TRAINING["val_split"]) * n)
test_idx = indices[val_end:]

test_loader = DataLoader(Subset(dataset, test_idx), batch_size=32, shuffle=False)
print(f"Test set: {len(test_idx)} samples")

## Run Predictions on Test Set

In [ ]:
all_preds = []
all_labels = []
all_probs = []

model.eval()
with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(DEVICE)
        outputs = model(images)
        probs = torch.softmax(outputs, dim=1)

        all_preds.extend(outputs.argmax(dim=1).cpu().numpy())
        all_labels.extend(labels.numpy())
        all_probs.extend(probs.cpu().numpy())

all_preds = np.array(all_preds)
all_labels = np.array(all_labels)
all_probs = np.array(all_probs)

accuracy = 100.0 * np.mean(all_preds == all_labels)
print(f"Test accuracy: {accuracy:.2f}%")

## Confusion Matrix

In [ ]:
def plot_confusion_matrix(labels, preds, class_names):
    """Plot a normalized confusion matrix."""
    n_classes = len(class_names)
    cm = np.zeros((n_classes, n_classes), dtype=int)
    for true, pred in zip(labels, preds):
        cm[true, pred] += 1

    # Normalize per row (per true class)
    cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)
    cm_norm = np.nan_to_num(cm_norm)

    fig, axes = plt.subplots(1, 2, figsize=(16, 6))

    # Raw counts
    im0 = axes[0].imshow(cm, cmap="Blues")
    axes[0].set_title("Confusion Matrix (counts)")
    for i in range(n_classes):
        for j in range(n_classes):
            axes[0].text(j, i, str(cm[i, j]), ha="center", va="center",
                        color="white" if cm[i, j] > cm.max() / 2 else "black")

    # Normalized
    im1 = axes[1].imshow(cm_norm, cmap="Blues", vmin=0, vmax=1)
    axes[1].set_title("Confusion Matrix (normalized)")
    for i in range(n_classes):
        for j in range(n_classes):
            axes[1].text(j, i, f"{cm_norm[i, j]:.2f}", ha="center", va="center",
                        color="white" if cm_norm[i, j] > 0.5 else "black")

    for ax in axes:
        ax.set_xticks(range(n_classes))
        ax.set_yticks(range(n_classes))
        ax.set_xticklabels(class_names, rotation=45, ha="right")
        ax.set_yticklabels(class_names)
        ax.set_xlabel("Predicted")
        ax.set_ylabel("True")

    plt.tight_layout()
    plt.show()

    return cm

cm = plot_confusion_matrix(all_labels, all_preds, CLASS_NAMES)

## Per-Class Metrics

In [ ]:
def per_class_metrics(labels, preds, class_names):
    """Compute precision, recall, F1 per class."""
    n_classes = len(class_names)
    print(f"{'Class':<16} {'Precision':>10} {'Recall':>10} {'F1':>10} {'Support':>10}")
    print("-" * 60)

    for i, name in enumerate(class_names):
        tp = np.sum((preds == i) & (labels == i))
        fp = np.sum((preds == i) & (labels != i))
        fn = np.sum((preds != i) & (labels == i))
        support = np.sum(labels == i)

        precision = tp / (tp + fp) if (tp + fp) > 0 else 0
        recall = tp / (tp + fn) if (tp + fn) > 0 else 0
        f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0

        print(f"{name:<16} {precision:>10.3f} {recall:>10.3f} {f1:>10.3f} {support:>10d}")

per_class_metrics(all_labels, all_preds, CLASS_NAMES)

## Confidence Distribution

How confident is the model on correct vs incorrect predictions?
Low-confidence correct predictions and high-confidence incorrect
predictions are both worth investigating.

In [ ]:
correct_mask = all_preds == all_labels
max_probs = np.max(all_probs, axis=1)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(max_probs[correct_mask], bins=30, alpha=0.7, color="green", label="Correct")
axes[0].hist(max_probs[~correct_mask], bins=30, alpha=0.7, color="red", label="Incorrect")
axes[0].set_xlabel("Max softmax probability")
axes[0].set_ylabel("Count")
axes[0].set_title("Prediction Confidence")
axes[0].legend()

# Per-class confidence
class_confidences = defaultdict(list)
for label, prob in zip(all_labels, max_probs):
    class_confidences[CLASS_NAMES[label]].append(prob)

data = [class_confidences[cls] for cls in CLASS_NAMES]
bp = axes[1].boxplot(data, labels=CLASS_NAMES, patch_artist=True)
colors_box = ["#FFB6C1", "#4169E1", "#FF0000", "#800080", "#C8C8C8"]
for patch, c in zip(bp["boxes"], colors_box):
    patch.set_facecolor(c)
axes[1].set_ylabel("Max softmax probability")
axes[1].set_title("Confidence by True Class")
axes[1].tick_params(axis="x", rotation=15)

plt.tight_layout()
plt.show()

## Failure Case Analysis

Visualize the misclassified patches. Look for patterns — are failures
clustered at class boundaries (e.g., vessel_h predicted as background_h)?

In [ ]:
def show_failures(dataset, test_idx, labels, preds, probs, max_show=20):
    """Display misclassified patches with true/predicted labels."""
    wrong_indices = np.where(labels != preds)[0]
    n_wrong = len(wrong_indices)
    print(f"\nTotal misclassified: {n_wrong} / {len(labels)} ({100*n_wrong/len(labels):.1f}%)")

    if n_wrong == 0:
        print("No failures to show!")
        return

    n_show = min(max_show, n_wrong)
    # Sort by confidence (most confident mistakes first — most concerning)
    wrong_probs = np.max(probs[wrong_indices], axis=1)
    sorted_idx = wrong_indices[np.argsort(-wrong_probs)][:n_show]

    cols = min(5, n_show)
    rows = (n_show + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(3.5 * cols, 4 * rows))
    if rows == 1:
        axes = axes.reshape(1, -1) if cols > 1 else np.array([[axes]])

    for i, idx in enumerate(sorted_idx):
        row, col = divmod(i, cols)
        ax = axes[row, col]

        # Get the actual dataset index
        ds_idx = test_idx[idx]
        img_tensor, _ = dataset[ds_idx]
        img = img_tensor.permute(1, 2, 0).numpy()
        # Undo normalization for display
        mean = np.array([0.485, 0.456, 0.406])
        std = np.array([0.229, 0.224, 0.225])
        img = img * std + mean
        img = np.clip(img, 0, 1)

        ax.imshow(img)
        true_name = CLASS_NAMES[labels[idx]]
        pred_name = CLASS_NAMES[preds[idx]]
        conf = probs[idx, preds[idx]]
        ax.set_title(f"True: {true_name}\nPred: {pred_name} ({conf:.2f})",
                    fontsize=9, color="red")
        ax.axis("off")

    # Hide empty axes
    for i in range(n_show, rows * cols):
        row, col = divmod(i, cols)
        axes[row, col].axis("off")

    plt.suptitle("Misclassified Patches (sorted by confidence)", fontsize=13)
    plt.tight_layout()
    plt.show()

show_failures(dataset, test_idx, all_labels, all_preds, all_probs)

## Key Takeaways

- [ ] Overall accuracy meets target (>85%)
- [ ] No class has disproportionately low recall
- [ ] Vessel classes (vessel_h, vessel_e) have high precision (few false positives)
- [ ] Failures make sense (borderline cases, not systematic errors)
- [ ] Model is well-calibrated (high confidence on correct, low on incorrect)